# IPA Reasoning Coach — Lakehouse Setup

**Idempotent** — safe to re-run. Creates `IPACoachLH` if it does not exist, then overwrites all 19 Delta tables.

**19 tables across 5 layers:**
- Layer 1: `SpeechVariety`, `PronunciationRule`, `RuleVariety`, `RuleConflict`, `PhoneticContext`
- Layer 2: `Transformation`, `ReasoningStep`, `TransformChain`
- Layer 3: `LexicalItem`, `Sentence`, `AnnotatedSegment`, `AudioReference`, `SentenceTag`
- Cross-cutting: `CommunicativeFactor`, `RuleFactorImpact`
- Layer 4: `MinimalPair`, `ShadowingPrompt`, `DrillSet`, `UserQuery`

In [ ]:
%pip install semantic-link --quiet --disable-pip-version-check

import requests
import sempy.fabric as fabric
from notebookutils import mssparkutils
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    BooleanType, FloatType
)

workspace_id = fabric.get_workspace_id()
token = mssparkutils.credentials.getToken('pbi')
headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}
print(f'Workspace ID: {workspace_id}\n')

print('Checking for IPACoachLH...')
resp = requests.get(
    f'https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/lakehouses',
    headers=headers
)
lakehouses = resp.json().get('value', [])
lh = next((l for l in lakehouses if l['displayName'] == 'IPACoachLH'), None)

if lh:
    lakehouse_id = lh['id']
    print(f'   Found existing IPACoachLH (id: {lakehouse_id})')
else:
    print('   Creating IPACoachLH...')
    lakehouse_id = fabric.create_lakehouse(
        display_name='IPACoachLH',
        description='IPA Reasoning Coach phonological knowledge base',
        enable_schema=False
    )
    print(f'   Created IPACoachLH (id: {lakehouse_id})')

lakehouse_tables_path = (
    f'abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}/Tables'
)
spark = SparkSession.builder.getOrCreate()

print(f'\nPath : {lakehouse_tables_path}')
print('=' * 60)
print('Infrastructure ready!')
print('=' * 60)

## Layer 1 — Phonological rules

Tables: `SpeechVariety`, `PronunciationRule`, `RuleVariety`, `RuleConflict`, `PhoneticContext`

In [ ]:
print('Writing SpeechVariety...')
data = [
    ('var_genam', 'General American',      'United States', 'Midwest/West',    'Most common US broadcast standard'),
    ('var_rp',    'Received Pronunciation', 'United Kingdom','Southern England','Traditional BBC English'),
    ('var_aue',   'Australian English',     'Australia',     'General',         'Placeholder for future expansion'),
    ('var_cane',  'Canadian English',       'Canada',        'General',         'Placeholder for future expansion'),
]
schema = StructType([
    StructField('variety_id',   StringType(), True),
    StructField('variety_name', StringType(), True),
    StructField('country',      StringType(), True),
    StructField('region',       StringType(), True),
    StructField('notes',        StringType(), True),
])
spark.createDataFrame(data, schema).write.mode('overwrite').format('delta').save(
    f'{lakehouse_tables_path}/SpeechVariety'
)
print('   SpeechVariety  (4 rows)')

In [ ]:
print('Writing PronunciationRule...')
data = [
    ('r01','flapping','connected_speech',
     'Intervocalic /t/ and /d/ realized as alveolar tap [ɾ] in unstressed syllables',
     'The /t/ sounds like a quick /d/ — butter sounds like budder',
     '/t/ or /d/ between two vowels, second unstressed','high','B1',False,0.97),
    ('r02','t_glottalization','connected_speech',
     'Replacement of /t/ with glottal stop [ʔ] before syllabic /n/ or at syllable coda',
     'The /t/ disappears and becomes a tiny catch in the throat',
     '/t/ before syllabic /n/ or in final coda (RP dominant)','high','B2',True,0.89),
    ('r03','h_dropping','connected_speech',
     'Deletion of /h/ in unstressed function words (him, her, his, have)',
     'The H vanishes — tell him sounds like tell im',
     '/h/ in unstressed pronoun or auxiliary','high','B1',True,0.93),
    ('r04','linking_r','connected_speech',
     'Insertion of /r/ between a word ending in /ə/ or /ɜː/ and a following vowel (RP)',
     'A ghost R appears between vowels in RP — law and order becomes law-r-and order',
     'Non-rhotic vowel + word boundary + vowel-initial word','high','B2',False,0.95),
    ('r05','intrusive_r','connected_speech',
     'Insertion of /r/ between any open vowel and following vowel even without historical R (RP)',
     'An R sneaks in where there was never one — idea of becomes idea-r-of',
     'Final open vowel + word boundary + vowel-initial word (RP)','medium','C1',True,0.82),
    ('r06','yod_coalescence','assimilation',
     'Fusion of /t/ or /d/ with following /j/ to produce [tʃ] or [dʒ]',
     'did you becomes didja, don t you becomes doncha',
     '/t/ or /d/ at word boundary + /j/ initial word','high','B2',False,0.91),
    ('r07','assimilation_nasal','assimilation',
     'Place assimilation of alveolar /n/ to bilabial or velar before consonants',
     'in person becomes im person; in class becomes ing class',
     '/n/ before bilabial or velar stop','medium','C1',False,0.88),
    ('r08','weak_forms','weak_forms',
     'Reduction of function words (and, of, to, for, from, than, that) to schwa-based forms',
     'and becomes /ən/, of becomes /əv/, to becomes /tə/ in connected speech',
     'Unstressed function word in connected speech','high','B1',False,0.98),
    ('r09','contraction','weak_forms',
     'Phonological reduction of auxiliaries: will to ll, would to d, have to ve, is to s',
     'Full forms shrink — I will becomes I ll, she is becomes she s',
     'Auxiliary verb in unstressed position','high','A2',False,0.99),
    ('r10','gonna_reduction','weak_forms',
     'Reduction of going to to [ɡʌnə] in informal speech',
     'going to becomes gonna — very common in fast casual speech',
     'going to + verb infinitive in informal register','high','B1',True,0.96),
    ('r11','wanna_reduction','weak_forms',
     'Reduction of want to to [ˈwɑnə] in informal speech',
     'want to becomes wanna — ubiquitous in American English',
     'want to in informal register','high','B1',True,0.96),
    ('r12','gotta_reduction','weak_forms',
     'Reduction of got to to [ˈɡɑɾə] in informal speech',
     'got to becomes gotta — implies obligation in casual speech',
     'got to in informal register','high','B1',True,0.95),
    ('r13','shoulda_reduction','weak_forms',
     'Reduction of should have to [ˈʃʊdə] — modal + have collapse',
     'should have becomes shoulda; similarly woulda coulda',
     'should/would/could + have in informal register','medium','B2',True,0.94),
    ('r14','elision_t_cluster','elision',
     'Deletion of /t/ in consonant clusters especially before another consonant',
     'The middle T disappears — next day becomes nex day, last call becomes las call',
     '/t/ in final consonant cluster before consonant','high','B2',False,0.90),
    ('r15','vowel_reduction','vowel_reduction',
     'Reduction of unstressed vowels to schwa /ə/ or close variants',
     'Unstressed vowels go gray and flat — the /ðiː/ becomes /ðə/',
     'Unstressed vowel in non-final position','high','B1',False,0.98),
    ('r16','smoothing','vowel_reduction',
     'Compression of diphthongs and triphthongs in rapid speech (RP)',
     'Vowel sequences squish together — fire /faɪər/ becomes [faː]',
     'Diphthong or triphthong in rapid RP speech','medium','C1',True,0.80),
]
schema = StructType([
    StructField('rule_id',          StringType(),  True),
    StructField('rule_name',        StringType(),  True),
    StructField('category',         StringType(),  True),
    StructField('description',      StringType(),  True),
    StructField('informal_desc',    StringType(),  True),
    StructField('trigger_pattern',  StringType(),  True),
    StructField('frequency',        StringType(),  True),
    StructField('difficulty_cefr',  StringType(),  True),
    StructField('is_optional',      BooleanType(), True),
    StructField('confidence_score', FloatType(),   True),
])
spark.createDataFrame(data, schema).write.mode('overwrite').format('delta').save(
    f'{lakehouse_tables_path}/PronunciationRule'
)
print('   PronunciationRule  (16 rows)')

In [ ]:
print('Writing RuleVariety...')
data = [
    ('r01','var_genam','primary',  'Core GenAm feature'),
    ('r01','var_rp',   'absent',   'RP uses /t/ or glottal stop instead'),
    ('r02','var_rp',   'primary',  'Dominant in RP especially pre-/n/'),
    ('r02','var_genam','secondary','Less systematic than RP'),
    ('r03','var_genam','secondary','Mainly in unstressed pronouns'),
    ('r03','var_rp',   'secondary','Common in casual RP'),
    ('r04','var_rp',   'primary',  'Non-rhotic: linking R obligatory'),
    ('r04','var_genam','absent',   'GenAm is rhotic, no linking R needed'),
    ('r05','var_rp',   'primary',  'Extension of linking R to non-etymological contexts'),
    ('r05','var_genam','absent',   'Not present in rhotic varieties'),
    ('r06','var_genam','primary',  'Very frequent in fast casual speech'),
    ('r06','var_rp',   'primary',  'Common in casual RP'),
    ('r07','var_genam','primary',  None),
    ('r07','var_rp',   'primary',  None),
    ('r08','var_genam','primary',  None),
    ('r08','var_rp',   'primary',  None),
    ('r09','var_genam','primary',  None),
    ('r09','var_rp',   'primary',  None),
    ('r10','var_genam','primary',  'Most frequent in GenAm informal'),
    ('r10','var_rp',   'secondary','Less frequent but present'),
    ('r11','var_genam','primary',  None),
    ('r11','var_rp',   'secondary',None),
    ('r12','var_genam','primary',  None),
    ('r12','var_rp',   'secondary',None),
    ('r13','var_genam','primary',  None),
    ('r13','var_rp',   'primary',  None),
    ('r14','var_genam','primary',  None),
    ('r14','var_rp',   'primary',  None),
    ('r15','var_genam','primary',  None),
    ('r15','var_rp',   'primary',  None),
    ('r16','var_rp',   'primary',  'Characteristic of rapid RP'),
    ('r16','var_genam','absent',   'Not systematic in GenAm'),
]
schema = StructType([
    StructField('rule_id',    StringType(), True),
    StructField('variety_id', StringType(), True),
    StructField('status',     StringType(), True),
    StructField('notes',      StringType(), True),
])
spark.createDataFrame(data, schema).write.mode('overwrite').format('delta').save(
    f'{lakehouse_tables_path}/RuleVariety'
)
print('   RuleVariety  (32 rows)')

In [ ]:
print('Writing RuleConflict...')
data = [
    ('cf01','r01','r02','r01',
     '/t/ between vowels, second unstressed',
     'Flapping takes precedence over t-glottalization in intervocalic position in GenAm',
     'var_genam', 0.92),
    ('cf02','r02','r01','r02',
     '/t/ before syllabic /n/ (button, mountain)',
     'T-glottalization wins before syllabic /n/ even in GenAm: flapping does not cross syllabic boundary',
     'var_genam', 0.90),
    ('cf03','r04','r05','r04',
     'Word-final non-rhotic vowel followed by vowel-initial word',
     'Linking-R has etymological basis and applies first; intrusive-R fills non-etymological gaps',
     'var_rp', 0.88),
    ('cf04','r06','r15','r06',
     'Stressed syllable with /t/ + /j/ cluster',
     'Yod coalescence applies before vowel reduction because it operates at consonant level first',
     None, 0.85),
    ('cf05','r08','r09','r09',
     'Auxiliary in unstressed position',
     'Contraction (r09) is a morphological process applied before phonological weak-form reduction (r08)',
     None, 0.87),
    ('cf06','r14','r07','r14',
     'Final /nt/ cluster before consonant (want more)',
     'Cluster elision of /t/ applies first; nasal assimilation is then fed by the result',
     None, 0.83),
]
schema = StructType([
    StructField('conflict_id',       StringType(), True),
    StructField('rule_a',            StringType(), True),
    StructField('rule_b',            StringType(), True),
    StructField('priority_rule',     StringType(), True),
    StructField('context_condition', StringType(), True),
    StructField('resolution_reason', StringType(), True),
    StructField('variety_id',        StringType(), True),
    StructField('confidence_score',  FloatType(),  True),
])
spark.createDataFrame(data, schema).write.mode('overwrite').format('delta').save(
    f'{lakehouse_tables_path}/RuleConflict'
)
print('   RuleConflict  (6 rows)')

print('Writing PhoneticContext...')
data = [
    ('ctx01','r01','Intervocalic position, second syllable unstressed','vowel','vowel',   'coda',   'unstressed',False,'butter', 0.97),
    ('ctx02','r01','Word-initial stressed syllable: flapping BLOCKED',  None,   None,    'onset',  'stressed',  True, 'tiger',  0.95),
    ('ctx03','r02','Before syllabic /n/',                               None,   'n',     'coda',   'any',       False,'button', 0.91),
    ('ctx04','r02','Word-final coda position (RP)',                     None,   None,    'coda',   'any',       False,'that',   0.88),
    ('ctx05','r03','Unstressed pronoun after verb',                     'V',    None,    'onset',  'unstressed',False,'tell him',0.93),
    ('ctx06','r04','Non-rhotic vowel at word end + vowel-initial word', None,   'V',     'coda',   'any',       False,'law and',0.96),
    ('ctx07','r06','/t/ or /d/ at word boundary before /j/',           None,   'j',     'coda',   'any',       False,'did you',0.91),
    ('ctx08','r14','Consonant cluster with /t/ before consonant',       'C',    'C',     'coda',   'any',       False,'next day',0.90),
    ('ctx09','r15','Unstressed vowel in non-final syllable',            None,   None,    'nucleus','unstressed',False,'banana', 0.98),
    ('ctx10','r08','Function word in unstressed syntactic position',    None,   None,    'any',    'unstressed',False,'and',    0.98),
]
schema = StructType([
    StructField('context_id',        StringType(),  True),
    StructField('rule_id',           StringType(),  True),
    StructField('environment_desc',  StringType(),  True),
    StructField('preceding_sound',   StringType(),  True),
    StructField('following_sound',   StringType(),  True),
    StructField('syllable_position', StringType(),  True),
    StructField('stress_condition',  StringType(),  True),
    StructField('blocks_rule',       BooleanType(), True),
    StructField('example_word',      StringType(),  True),
    StructField('confidence_score',  FloatType(),   True),
])
spark.createDataFrame(data, schema).write.mode('overwrite').format('delta').save(
    f'{lakehouse_tables_path}/PhoneticContext'
)
print('   PhoneticContext  (10 rows)')

## Layer 2 — Transformations

Tables: `Transformation`, `ReasoningStep`, `TransformChain`

In [ ]:
print('Writing Transformation...')
data = [
    ('t01','r01','var_genam','butter',     'ˈbʌtər',  'ˈbʌɾər',  'substitution', True,  0.97, 'Intervocalic /t/ to tap'),
    ('t02','r01','var_genam','city',       'ˈsɪti',   'ˈsɪɾi',   'substitution', True,  0.96, 'Intervocalic /t/ to tap'),
    ('t03','r01','var_genam','water',      'ˈwɔːtər', 'ˈwɔːɾər', 'substitution', True, 0.97, None),
    ('t04','r02','var_rp',   'button',     'ˈbʌtən',  'ˈbʌʔn̩',   'substitution', True,  0.91, '/t/ to glottal stop before syllabic /n/'),
    ('t05','r02','var_rp',   'that',       'ðæt',     'ðæʔ',     'substitution', False, 0.85, 'Word-final glottalization'),
    ('t06','r03','var_genam','tell him',   'tɛl hɪm', 'tɛl ɪm',  'deletion',     False, 0.93, 'H-drop in unstressed pronoun'),
    ('t07','r03','var_rp',   'tell her',   'tɛl hɜː', 'tɛl ɜː',  'deletion',     False, 0.91, None),
    ('t08','r04','var_rp',   'law and',    'lɔː ænd', 'lɔːr ænd', 'insertion',   True,  0.95, 'Linking-R inserted'),
    ('t09','r05','var_rp',   'idea of',    'aɪˈdɪə ɒv','aɪˈdɪər ɒv','insertion', False, 0.82, 'Intrusive-R, no etymological R'),
    ('t10','r06','var_genam','did you',    'dɪd juː', 'dɪdʒu',   'fusion',       False, 0.91, 'Yod coalescence /d+j/ to [dZ]'),
    ('t11','r06','var_genam','don t you',  'dəʊnt juː','dəʊntʃu', 'fusion', False, 0.90, 'Yod coalescence /t+j/ to [tS]'),
    ('t12','r08','var_genam','and',        'ænd',     'ən',       'reduction',    True,  0.98, 'Weak form of and'),
    ('t13','r08','var_genam','of',         'ɒv',      'əv',       'reduction',    True,  0.98, 'Weak form of of'),
    ('t14','r08','var_genam','to',         'tuː',     'tə',       'reduction',    True,  0.97, 'Weak form of to'),
    ('t15','r08','var_genam','for',        'fɔːr',    'fər',      'reduction',    True,  0.97, None),
    ('t16','r09','var_genam','I will',     'aɪ wɪl',  'aɪl',      'reduction',    False, 0.99, 'Contraction I ll'),
    ('t17','r09','var_genam','she is',     'ʃiː ɪz',  'ʃɪz',     'reduction',    False, 0.99, 'Contraction she s'),
    ('t18','r10','var_genam','going to',   'ˈɡoʊɪŋ tuː','ˈɡʌnə', 'reduction', False, 0.96, 'gonna'),
    ('t19','r11','var_genam','want to',    'wɒnt tuː','ˈwɑnə',    'reduction',    False, 0.96, 'wanna'),
    ('t20','r12','var_genam','got to',     'ɡɒt tuː', 'ˈɡɑɾə',   'reduction',    False, 0.95, 'gotta'),
    ('t21','r13','var_genam','should have','ʃʊd hæv', 'ˈʃʊdə',   'reduction',    False, 0.94, 'shoulda'),
    ('t22','r13','var_genam','would have', 'wʊd hæv', 'ˈwʊdə',   'reduction',    False, 0.94, 'woulda'),
    ('t23','r13','var_genam','could have', 'kʊd hæv', 'ˈkʊdə',   'reduction',    False, 0.94, 'coulda'),
    ('t24','r14','var_genam','next day',   'nɛkst deɪ','nɛks deɪ', 'deletion',     True,  0.90, '/t/ deleted in /kst/ cluster'),
    ('t25','r14','var_genam','last call',  'læst kɔːl','læs kɔːl', 'deletion', True,  0.90, '/t/ deleted in /st/ cluster before /k/'),
    ('t26','r15','var_genam','about',      'æˈbaʊt',  'əˈbaʊt',  'reduction',    True,  0.98, 'Initial unstressed vowel to schwa'),
    ('t27','r15','var_genam','banana',     'bəˈnænə', 'bəˈnænə', 'reduction', True, 0.99, 'Both unstressed vowels to schwa'),
]
schema = StructType([
    StructField('transform_id',     StringType(),  True),
    StructField('rule_id',          StringType(),  True),
    StructField('variety_id',       StringType(),  True),
    StructField('input_segment',    StringType(),  True),
    StructField('formal_ipa',       StringType(),  True),
    StructField('casual_ipa',       StringType(),  True),
    StructField('change_type',      StringType(),  True),
    StructField('is_obligatory',    BooleanType(), True),
    StructField('confidence_score', FloatType(),   True),
    StructField('notes',            StringType(),  True),
])
spark.createDataFrame(data, schema).write.mode('overwrite').format('delta').save(
    f'{lakehouse_tables_path}/Transformation'
)
print('   Transformation  (27 rows)')

In [ ]:
print('Writing ReasoningStep...')
# Full reasoning chain for: I ll talk to her later (GenAm)
data = [
    ('s01','t16',1,'detect',   'Detected I will: auxiliary in unstressed position, contraction applies','ɪ wɪl',    'aɪl',     'r09',0.99),
    ('s02','t14',2,'detect',   'Detected to as unstressed function word: weak form applies',           'tuː',     'tə',       'r08',0.97),
    ('s03','t07',3,'detect',   'Detected her in post-verbal unstressed position: H-drop applies',      'hɜːr',    'ɜːr',     'r03',0.91),
    ('s04','t01',4,'detect',   'Detected /t/ in later between two vowels, second syllable unstressed: flapping applies','ˈleɪtər','ˈleɪɾər','r01',0.97),
    ('s05','t01',5,'conflict_check','No conflict: flapping confirmed, not before syllabic /n/ or word-final coda','ˈleɪtər','ˈleɪɾər','r01',0.95),
    ('s06','t16',6,'apply',    'Applied contraction: I will becomes I ll /aɪl/',                  'aɪ wɪl',  'aɪl',     'r09',0.99),
    ('s07','t14',7,'apply',    'Applied weak form: to becomes /tə/',                              'tuː',     'tə',       'r08',0.97),
    ('s08','t07',8,'apply',    'Applied H-drop: her becomes /ɜːr/',                          'hɜːr',    'ɜːr',     'r03',0.91),
    ('s09','t01',9,'apply',    'Applied flapping: later /ˈleɪtər/ becomes /ˈleɪɾər/',  'ˈleɪtər','ˈleɪɾər','r01',0.97),
    ('s10','t16',10,'explain', 'Contraction is obligatory in informal speech; full form signals formality or emphasis',None, None,'r09',0.99),
    ('s11','t16',11,'verify',  'Final casual IPA: /aɪl tɔːk tə ɜːr ˈleɪɾər/ — 4 rules applied',None,None,None,0.96),
]
schema = StructType([
    StructField('step_id',          StringType(),  True),
    StructField('transform_id',     StringType(),  True),
    StructField('step_order',       IntegerType(), True),
    StructField('step_type',        StringType(),  True),
    StructField('step_description', StringType(),  True),
    StructField('ipa_before',       StringType(),  True),
    StructField('ipa_after',        StringType(),  True),
    StructField('rule_invoked',     StringType(),  True),
    StructField('confidence_score', FloatType(),   True),
])
spark.createDataFrame(data, schema).write.mode('overwrite').format('delta').save(
    f'{lakehouse_tables_path}/ReasoningStep'
)
print('   ReasoningStep  (11 rows)')

print('Writing TransformChain...')
data = [
    ('ch01','sent01','var_genam',4,
     'aɪ wɪl tɔːk tuː hɜːr ˈleɪtər',
     'aɪl tɔːk tə ɜːr ˈleɪɾər',
     'r09,r08,r03,r01', 0.95),
    ('ch02','sent02','var_genam',3,
     'ˈwɒt ər juː ˈɡoʊɪŋ tuː duː',
     'ˈwʌɾər jə ˈɡʌnə duː',
     'r01,r15,r10', 0.94),
    ('ch03','sent03','var_genam',3,
     'aɪ ʃʊd hæv noʊn ˈbɛtər',
     'aɪ ˈʃʊdə noʊn ˈbɛɾər',
     'r13,r01,r08', 0.93),
    ('ch04','sent04','var_rp',3,
     'ðə ˈaɪdɪə ɒv ɪt',
     'ðə ˈaɪdɪər ɒv ɪt',
     'r05,r08,r15', 0.88),
]
schema = StructType([
    StructField('chain_id',         StringType(),  True),
    StructField('sentence_id',      StringType(),  True),
    StructField('variety_id',       StringType(),  True),
    StructField('step_count',       IntegerType(), True),
    StructField('formal_ipa_full',  StringType(),  True),
    StructField('casual_ipa_full',  StringType(),  True),
    StructField('rules_applied',    StringType(),  True),
    StructField('confidence_score', FloatType(),   True),
])
spark.createDataFrame(data, schema).write.mode('overwrite').format('delta').save(
    f'{lakehouse_tables_path}/TransformChain'
)
print('   TransformChain  (4 rows)')

## Layer 3 — Examples corpus

Tables: `LexicalItem`, `Sentence`, `AnnotatedSegment`, `AudioReference`, `SentenceTag`

In [ ]:
print('Writing LexicalItem...')
data = [
    ('w01','going to',  'gonna',   1,  'B1','informal','aux', True,  'going to','r10'),
    ('w02','want to',   'wanna',   2,  'B1','informal','aux', True,  'want to', 'r11'),
    ('w03','got to',    'gotta',   3,  'B1','informal','aux', True,  'got to',  'r12'),
    ('w04','should have','shoulda',4,  'B2','informal','aux', True,  'should have','r13'),
    ('w05','would have','woulda',  5,  'B2','informal','aux', True,  'would have','r13'),
    ('w06','could have','coulda',  6,  'B2','informal','aux', True,  'could have','r13'),
    ('w07','have to',   'hafta',   7,  'B1','informal','aux', True,  'have to', 'r08'),
    ('w08','kind of',   'kinda',   8,  'B1','informal','adv', True,  'kind of', 'r08'),
    ('w09','sort of',   'sorta',   9,  'B1','informal','adv', True,  'sort of', 'r08'),
    ('w10','out of',    'outta',   10, 'B1','informal','prep',True,  'out of',  'r08'),
    ('w11','and',       'and',     11, 'A1','neutral', 'conj',False, None,      'r08'),
    ('w12','of',        'of',      12, 'A1','neutral', 'prep',False, None,      'r08'),
    ('w13','to',        'to',      13, 'A1','neutral', 'prep',False, None,      'r08'),
    ('w14','butter',    'butter',  100,'A1','neutral', 'noun',False, None,      'r01'),
    ('w15','later',     'later',   150,'A2','neutral', 'adv', False, None,      'r01'),
    ('w16','water',     'water',   50, 'A1','neutral', 'noun',False, None,      'r01'),
    ('w17','button',    'button',  200,'A1','neutral', 'noun',False, None,      'r02'),
    ('w18','mountain',  'mountain',250,'A1','neutral', 'noun',False, None,      'r02'),
]
schema = StructType([
    StructField('word_id',        StringType(),  True),
    StructField('lemma',          StringType(),  True),
    StructField('surface_form',   StringType(),  True),
    StructField('frequency_rank', IntegerType(), True),
    StructField('cefr_level',     StringType(),  True),
    StructField('register',       StringType(),  True),
    StructField('pos',            StringType(),  True),
    StructField('is_reduced',     BooleanType(), True),
    StructField('base_form',      StringType(),  True),
    StructField('rule_id',        StringType(),  True),
])
spark.createDataFrame(data, schema).write.mode('overwrite').format('delta').save(
    f'{lakehouse_tables_path}/LexicalItem'
)
print('   LexicalItem  (18 rows)')

In [ ]:
print('Writing Sentence...')
data = [
    ('sent01',"I'll talk to her later",         'informal','everyday',  'B1',5,'manual'),
    ('sent02',"What are you going to do?",       'informal','everyday',  'B1',6,'manual'),
    ('sent03',"I should have known better",      'informal','everyday',  'B2',5,'manual'),
    ('sent04',"The idea of it",                  'neutral', 'everyday',  'B1',4,'manual'),
    ('sent05',"Did you eat yet?",                'informal','everyday',  'B1',4,'manual'),
    ('sent06',"I want to go to the next meeting",'formal',  'business',  'B2',8,'manual'),
    ('sent07',"She could have told me",          'informal','everyday',  'B2',5,'manual'),
    ('sent08',"button, mountain, curtain",       'neutral', 'phonology', 'B2',3,'manual'),
    ('sent09',"Tell him to call me back",        'informal','everyday',  'B1',6,'manual'),
    ('sent10',"It was kind of a big deal",       'informal','everyday',  'B2',7,'manual'),
]
schema = StructType([
    StructField('sentence_id', StringType(),  True),
    StructField('raw_text',    StringType(),  True),
    StructField('register',    StringType(),  True),
    StructField('topic',       StringType(),  True),
    StructField('cefr_level',  StringType(),  True),
    StructField('word_count',  IntegerType(), True),
    StructField('source',      StringType(),  True),
])
spark.createDataFrame(data, schema).write.mode('overwrite').format('delta').save(
    f'{lakehouse_tables_path}/Sentence'
)
print('   Sentence  (10 rows)')

print('Writing AnnotatedSegment...')
data = [
    ('seg01','sent01',0, 2,  "I'll",     'r09','t16',1,0.99),
    ('seg02','sent01',7, 9,  'to',       'r08','t14',2,0.97),
    ('seg03','sent01',10,13, 'her',      'r03','t07',3,0.91),
    ('seg04','sent01',14,19, 'later',    'r01','t01',4,0.97),
    ('seg05','sent02',8, 17, 'going to', 'r10','t18',1,0.96),
    ('seg06','sent02',0, 4,  'What',     'r01',None, 1,0.90),
    ('seg07','sent03',2, 14, 'should have','r13','t21',1,0.94),
    ('seg08','sent03',19,25, 'better',   'r01','t01',2,0.97),
    ('seg09','sent04',4, 11, 'idea of',  'r05',None, 1,0.82),
    ('seg10','sent05',0, 6,  'Did you',  'r06','t10',1,0.91),
]
schema = StructType([
    StructField('segment_id',      StringType(),  True),
    StructField('sentence_id',     StringType(),  True),
    StructField('span_start',      IntegerType(), True),
    StructField('span_end',        IntegerType(), True),
    StructField('orthographic',    StringType(),  True),
    StructField('rule_id',         StringType(),  True),
    StructField('transform_id',    StringType(),  True),
    StructField('priority',        IntegerType(), True),
    StructField('confidence_score',FloatType(),   True),
])
spark.createDataFrame(data, schema).write.mode('overwrite').format('delta').save(
    f'{lakehouse_tables_path}/AnnotatedSegment'
)
print('   AnnotatedSegment  (10 rows)')

print('Writing AudioReference...')
data = [
    ('a01','sent01','https://youglish.com/pronounce/I+will+talk+to+her+later/english',
     'YouGlish','var_genam','any','natural',0.90),
    ('a02','sent02','https://youglish.com/pronounce/going+to/english',
     'YouGlish','var_genam','any','fast',   0.88),
    ('a03','sent03','https://youglish.com/pronounce/should+have+known/english',
     'YouGlish','var_genam','any','natural',0.87),
]
schema = StructType([
    StructField('audio_id',        StringType(), True),
    StructField('sentence_id',     StringType(), True),
    StructField('url',             StringType(), True),
    StructField('source_name',     StringType(), True),
    StructField('speaker_variety', StringType(), True),
    StructField('speaker_gender',  StringType(), True),
    StructField('speech_rate',     StringType(), True),
    StructField('confidence_score',FloatType(),  True),
])
spark.createDataFrame(data, schema).write.mode('overwrite').format('delta').save(
    f'{lakehouse_tables_path}/AudioReference'
)
print('   AudioReference  (3 rows)')

print('Writing SentenceTag...')
data = [
    ('tg01','sent01','rule_focus','flapping'),
    ('tg02','sent01','rule_focus','h_dropping'),
    ('tg03','sent01','rule_focus','contraction'),
    ('tg04','sent02','rule_focus','gonna'),
    ('tg05','sent03','rule_focus','shoulda'),
    ('tg06','sent04','rule_focus','intrusive_r'),
    ('tg07','sent05','rule_focus','yod_coalescence'),
    ('tg08','sent01','difficulty','intermediate'),
    ('tg09','sent03','difficulty','upper_intermediate'),
]
schema = StructType([
    StructField('tag_id',      StringType(), True),
    StructField('sentence_id', StringType(), True),
    StructField('tag_type',    StringType(), True),
    StructField('tag_value',   StringType(), True),
])
spark.createDataFrame(data, schema).write.mode('overwrite').format('delta').save(
    f'{lakehouse_tables_path}/SentenceTag'
)
print('   SentenceTag  (9 rows)')

## Cross-cutting — Communicative reasoning

Tables: `CommunicativeFactor`, `RuleFactorImpact`

These tables allow the agent to explain **why** a speaker chose a form, not just **what** transformation happened.

In [ ]:
print('Writing CommunicativeFactor...')
data = [
    ('f01','formality',       'social',  'Degree of formality of the communicative situation', True),
    ('f02','speech_rate',     'phonetic','How fast the speaker is speaking',                   True),
    ('f03','emphasis',        'pragmatic','Whether the speaker is emphasizing a word',          True),
    ('f04','emotional_state', 'pragmatic','Emotional arousal affects articulation precision',  False),
    ('f05','regional_identity','social', 'Speaker performing or signaling regional identity',  False),
    ('f06','audience_design', 'pragmatic','Speaker adapts to perceived audience',               False),
    ('f07','fatigue',         'phonetic','Articulatory effort decreases with fatigue',          False),
    ('f08','genre',           'social',  'Discourse genre: conversation, lecture, broadcast',  True),
]
schema = StructType([
    StructField('factor_id',   StringType(),  True),
    StructField('factor_name', StringType(),  True),
    StructField('category',    StringType(),  True),
    StructField('description', StringType(),  True),
    StructField('measurable',  BooleanType(), True),
])
spark.createDataFrame(data, schema).write.mode('overwrite').format('delta').save(
    f'{lakehouse_tables_path}/CommunicativeFactor'
)
print('   CommunicativeFactor  (8 rows)')

print('Writing RuleFactorImpact...')
data = [
    ('i01','r08','f01','decreases','Higher formality: speakers use full forms of and, of, to',0.95),
    ('i02','r08','f02','increases','Faster speech: weak forms more likely to apply',           0.96),
    ('i03','r08','f03','blocks',   'Emphasis on a function word overrides weak form',          0.92),
    ('i04','r10','f01','decreases','Informal reductions absent in formal or written register', 0.97),
    ('i05','r10','f02','increases','Fast speech rate triggers gonna reduction more systematically',0.94),
    ('i06','r10','f05','increases','GenAm regional identity correlates with higher gonna frequency',0.85),
    ('i07','r11','f01','decreases','want to survives in formal speech; wanna is informal marker',0.97),
    ('i08','r12','f01','decreases','got to survives in formal speech; gotta is informal marker', 0.97),
    ('i09','r13','f01','decreases','Modal reduction signals informality and low planning',     0.96),
    ('i10','r13','f04','increases','Emotional speech (regret, frustration) strongly with shoulda',0.88),
    ('i11','r13','f02','increases','Faster speech accelerates modal + have fusion',            0.91),
    ('i12','r01','f01','decreases','Formal speech maintains /t/: flapping is informal marker in GenAm',0.89),
    ('i13','r01','f02','increases','Faster speech rate increases flapping frequency',          0.93),
    ('i14','r01','f05','increases','GenAm regional identity associated with consistent flapping',0.87),
    ('i15','r03','f01','decreases','H-drop in pronouns less likely in careful or formal speech',0.88),
    ('i16','r03','f02','increases','Fast connected speech systematically drops H in unstressed pronouns',0.94),
    ('i17','r06','f01','decreases','Formal speech maintains /t+j/ and /d+j/ separately',     0.86),
    ('i18','r06','f02','triggers', 'Fast speech is primary trigger for yod coalescence at word boundaries',0.92),
    ('i19','r04','f08','increases','Broadcast RP maintains linking-R; casual RP may drop it', 0.83),
    ('i20','r05','f01','decreases','Intrusive-R is stigmatized; formal speakers suppress it', 0.80),
    ('i21','r02','f05','increases','Estuary/London regional identity correlates with glottalization',0.84),
    ('i22','r02','f01','decreases','Formal RP speakers maintain full /t/ where possible',     0.82),
    ('i23','r09','f03','blocks',   'Emphatic stress on auxiliary prevents contraction: I WILL go vs I ll go',0.98),
    ('i24','r09','f01','decreases','Formal speech prefers full auxiliary forms',               0.91),
    ('i25','r14','f02','increases','Cluster elision much more frequent at fast speech rates', 0.93),
    ('i26','r14','f07','increases','Fatigue reduces articulatory precision, increasing elision',0.79),
]
schema = StructType([
    StructField('impact_id',       StringType(), True),
    StructField('rule_id',         StringType(), True),
    StructField('factor_id',       StringType(), True),
    StructField('effect',          StringType(), True),
    StructField('explanation',     StringType(), True),
    StructField('confidence_score',FloatType(),  True),
])
spark.createDataFrame(data, schema).write.mode('overwrite').format('delta').save(
    f'{lakehouse_tables_path}/RuleFactorImpact'
)
print('   RuleFactorImpact  (26 rows)')

## Layer 4 — Practice and output

Tables: `MinimalPair` (seeded), `ShadowingPrompt`, `DrillSet`, `UserQuery` (empty — populated at runtime by the agent)

In [ ]:
print('Writing MinimalPair...')
data = [
    ('mp01','butter (careful)', 'ˈbʌtər', 'butter (casual)', 'ˈbʌɾər',
     'formal_vs_casual', 'r01',None,'var_genam','Flapping in casual GenAm'),
    ('mp02','button (GenAm)',   'ˈbʌtən',  'button (RP)',     'ˈbʌʔn̩',
     'variety',           'r01','r02',None,'Flapping vs glottalization across varieties'),
    ('mp03','did you (careful)','ˈdɪd juː','did you (casual)', 'dɪdʒu',
     'formal_vs_casual',  'r06',None,'var_genam','Yod coalescence in fast speech'),
    ('mp04','I will go',       'aɪ wɪl ɡoʊ','I ll go',   'aɪl ɡoʊ',
     'full_vs_contracted', 'r09',None,'var_genam','Contraction blocked when emphatic'),
    ('mp05','law and order (GenAm)','lɔː ænd ˈɔːrdər',
     'law and order (RP)','lɔːr ænd ˈɔːdə',
     'variety','r04',None,None,'Linking-R in RP, absent in GenAm'),
    ('mp06','should have gone','ʃʊd hæv ɡɒn','shoulda gone','ˈʃʊdə ɡɒn',
     'formal_vs_casual',  'r13',None,'var_genam','Modal reduction — regret context'),
]
schema = StructType([
    StructField('pair_id',      StringType(), True),
    StructField('item_a',       StringType(), True),
    StructField('ipa_a',        StringType(), True),
    StructField('item_b',       StringType(), True),
    StructField('ipa_b',        StringType(), True),
    StructField('contrast_type',StringType(), True),
    StructField('rule_a',       StringType(), True),
    StructField('rule_b',       StringType(), True),
    StructField('variety_id',   StringType(), True),
    StructField('explanation',  StringType(), True),
])
spark.createDataFrame(data, schema).write.mode('overwrite').format('delta').save(
    f'{lakehouse_tables_path}/MinimalPair'
)
print('   MinimalPair  (6 rows)')

# Empty tables populated at agent runtime
empty_tables = {
    'ShadowingPrompt': StructType([
        StructField('prompt_id',      StringType(), True),
        StructField('sentence_id',    StringType(), True),
        StructField('variety_id',     StringType(), True),
        StructField('chain_id',       StringType(), True),
        StructField('focus_rules',    StringType(), True),
        StructField('instruction',    StringType(), True),
        StructField('formal_ipa',     StringType(), True),
        StructField('casual_ipa',     StringType(), True),
        StructField('difficulty_cefr',StringType(), True),
    ]),
    'DrillSet': StructType([
        StructField('drill_id',        StringType(), True),
        StructField('rule_id',         StringType(), True),
        StructField('variety_id',      StringType(), True),
        StructField('drill_name',      StringType(), True),
        StructField('instructions',    StringType(), True),
        StructField('items',           StringType(), True),
        StructField('difficulty_cefr', StringType(), True),
    ]),
    'UserQuery': StructType([
        StructField('query_id',      StringType(),  True),
        StructField('session_id',    StringType(),  True),
        StructField('input_text',    StringType(),  True),
        StructField('detected_rules',StringType(),  True),
        StructField('variety_used',  StringType(),  True),
        StructField('agent_response',StringType(),  True),
        StructField('feedback_score',IntegerType(), True),
    ]),
}
for tname, tschema in empty_tables.items():
    spark.createDataFrame([], tschema).write.mode('overwrite').format('delta').save(
        f'{lakehouse_tables_path}/{tname}'
    )
    print(f'   {tname}  (empty, populated at runtime)')

## Validation — row counts and path check

Reads each Delta table from the lakehouse path and prints row counts. All counts must be non-zero except the three runtime tables.

In [ ]:
tables = [
    ('SpeechVariety',       4),
    ('PronunciationRule',   16),
    ('RuleVariety',         32),
    ('RuleConflict',        6),
    ('PhoneticContext',     10),
    ('Transformation',      27),
    ('ReasoningStep',       11),
    ('TransformChain',      4),
    ('LexicalItem',         18),
    ('Sentence',            10),
    ('AnnotatedSegment',    10),
    ('AudioReference',      3),
    ('SentenceTag',         9),
    ('CommunicativeFactor', 8),
    ('RuleFactorImpact',    26),
    ('MinimalPair',         6),
    ('ShadowingPrompt',     0),
    ('DrillSet',            0),
    ('UserQuery',           0),
]

print('=' * 50)
print(f'{"Table":<25} {"Expected":>8} {"Actual":>8} {"OK?":>5}')
print('-' * 50)
all_ok = True
for tname, expected in tables:
    actual = spark.read.format('delta').load(f'{lakehouse_tables_path}/{tname}').count()
    ok = actual == expected
    if not ok:
        all_ok = False
    marker = 'OK' if ok else 'FAIL'
    print(f'{tname:<25} {expected:>8} {actual:>8} {marker:>5}')
print('=' * 50)
if all_ok:
    print('\nAll tables validated. Lakehouse ready for Data Agent.')
else:
    print('\nSome tables have unexpected counts. Check output above.')